# `aidp-exacs` live test — IAM DB-Token (only IAM-enabled clusters)
**Live-test row 5.** Skip if the cluster is on classic auth.


In [ ]:
import sys, os
# Adjust this if you've uploaded the plugin scripts/ dir elsewhere.
sys.path.insert(0, '/Workspace/Shared/oracle_ai_data_platform_connectors/scripts')


In [ ]:
from oracle_ai_data_platform_connectors.auth import generate_db_token
from oracle_ai_data_platform_connectors.jdbc import build_oracle_jdbc_url, spark_jdbc_options_dbtoken

token_dir = generate_db_token(os.environ['EXACS_COMPARTMENT_OCID'], target_dir='/tmp/dbcred_exacs')
url = build_oracle_jdbc_url(host=os.environ['EXACS_HOST'], port=1522, service_name=os.environ['EXACS_SERVICE_NAME'])
opts = spark_jdbc_options_dbtoken(url=url, token_dir=token_dir)


In [ ]:
df = spark.read.format('jdbc').options(**opts).option('dbtable', os.environ['EXACS_TABLE_FOR_TEST']).load()
df.show(5)


In [ ]:
# Live-test driver parses this final cell's stdout for the JSON summary.
import json, time
summary = {
    'connector': 'aidp-exacs',
    'auth': 'dbtoken',
    'rows': int(df.count()),
    'schema': sorted([f.name for f in df.schema.fields]),
    'timestamp_utc': int(time.time()),
}
print('AIDP_LIVE_TEST_RESULT_BEGIN')
print(json.dumps(summary, indent=2))
print('AIDP_LIVE_TEST_RESULT_END')
